# Web Automation and Data Ingestion Lab - Student Notebook

This lab is the hands-on companion to the workflow automation lesson.

How to use it:
- Complete each TODO cell.
- Run the assert check cell after each exercise.
- When the assert cell passes, your output matches the expected result.
- If an assert fails, read the message and inspect your selectors, variable names, and output shape.


## Lab Setup

The lab uses the same local demo website as the lesson. This setup cell finds the demo HTML files, imports the tools we need, and creates output folders for raw and processed data.


In [ ]:
from pathlib import Path
import csv
import json
import logging
import socket
import threading
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("web-ingestion-lesson")

BASE_DIR = Path.cwd()
DEMO_CANDIDATES = [BASE_DIR / "../IntermediatePython/web_automation_demo", BASE_DIR / "IntermediatePython" / "web_automation_demo", BASE_DIR / "web_automation_demo"]
DEMO_DIR = next((path.resolve() for path in DEMO_CANDIDATES if path.exists()), None)

if DEMO_DIR is None:
    raise FileNotFoundError(
        "Could not find web_automation_demo. Expected it under IntermediatePython/web_automation_demo."
    )

DATA_DIR = BASE_DIR / "data" / "web_ingestion"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
LOG_DIR = BASE_DIR / "logs"

for directory in [RAW_DIR, PROCESSED_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Demo directory:", DEMO_DIR)
print("Processed output directory:", PROCESSED_DIR)


## Start the Local Demo Website

The local website gives us stable pages to practice against. Students can focus on selectors, parsing, waits, and validation without depending on an external website.


In [ ]:

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass


def start_demo_server(directory: Path):
    port = find_free_port()
    handler = lambda *args, **kwargs: QuietHandler(*args, directory=str(directory), **kwargs)
    server = ThreadingHTTPServer(("127.0.0.1", port), handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server, f"http://127.0.0.1:{port}"


demo_server, demo_base_url = start_demo_server(DEMO_DIR)
static_url = f"{demo_base_url}/products_static.html"
dynamic_url = f"{demo_base_url}/products_dynamic.html"

print("Static demo:", static_url)
print("Dynamic demo:", dynamic_url)


## Mini Exercise - Inspect and Identify Elements

Before writing extraction code, practice identifying the repeated container and the child fields. This mirrors the inspection process used on real websites.

**Objective:** inspect the local product page and decide which selectors you would use.

Open `products_static.html` in the browser or inspect the page source. Write your guesses in a note before moving to Exercise A.

Tasks:
1. Identify the repeated product card container.
2. Identify the product title selector.
3. Identify the product price selector.
4. Identify the product link selector.
5. Decide whether Beautiful Soup is enough for this page.

Hints:
- A repeated container usually wraps one complete record.
- CSS class selectors start with `.`.
- Product names are visible text inside a heading element.
- Product links store their destination in an `href` attribute.
- If the product cards are already present in the raw HTML, Beautiful Soup is enough.


## Assert-Driven Practice Lab

These practice tasks use assertions as feedback. If the assert cell passes, the shape and content of the result match the expected output.

The goal is to practice the workflow progressively:

1. Inspect HTML and choose stable selectors.
2. Extract data from a small inline sample.
3. Fetch and parse the full static page.
4. Wrap parsing in a reusable function.
5. Validate and save a clean CSV.
6. Try Selenium only after the static workflow is comfortable.

Keep your code simple. Passing assertions matter more than writing clever code.


### Exercise A - Identify Selectors from HTML

**Objective:** identify the selectors needed to extract product data.

Complete the TODO cell by assigning CSS selector strings to the variables.

Hints:
- `card_selector` should identify the repeated parent container for one product.
- `name_selector` can be short because it will be used inside one card.
- `price_selector` should target the element containing the price text.
- `link_selector` should target the anchor element that stores the detail URL.
- Do not call `select()` in this exercise. Store selector strings only.


In [ ]:

# Exercise A TODO

card_selector = ""
name_selector = ""
price_selector = ""
link_selector = ""


In [ ]:
# Exercise A Checks
# These checks validate behavior without showing the exact selector answers.

exercise_a_html = """
<div class="product-card" data-product-id="001">
  <h2>Wireless Mouse</h2>
  <p class="price">$25.99</p>
  <a class="product-link" href="/products/mouse">View details</a>
</div>

<div class="product-card" data-product-id="002">
  <h2>Keyboard</h2>
  <p class="price">$45.00</p>
  <a class="product-link" href="/products/keyboard">View details</a>
</div>
"""

exercise_a_soup = BeautifulSoup(exercise_a_html, "html.parser")
exercise_a_cards = exercise_a_soup.select(card_selector)

assert card_selector, "Assign a selector for the repeated product card container."
assert name_selector, "Assign a selector for the product name."
assert price_selector, "Assign a selector for the product price."
assert link_selector, "Assign a selector for the product link."
assert len(exercise_a_cards) == 2, "The card selector should find exactly two product cards."

first_card = exercise_a_cards[0]
assert first_card.select_one(name_selector).get_text(strip=True) == "Wireless Mouse"
assert first_card.select_one(price_selector).get_text(strip=True) == "$25.99"
assert first_card.select_one(link_selector).get("href") == "/products/mouse"

print("Exercise A passed")


## Mini Exercise - Parse Static HTML with Beautiful Soup

This mini exercise uses a small inline HTML sample so the parent-container pattern is easy to see and test.

**Objective:** fill in the selectors so the loop finds each product card, then extracts the product name, price, and link URL from inside each card.

Hints:
- Start by selecting the repeated parent element.
- Once you are inside one card, use short child selectors.
- The visible link text is not the URL. Read the link attribute.


In [ ]:

html = """
<div class="product-card">
  <h2>Wireless Mouse</h2>
  <p class="price">$25.99</p>
  <a class="product-link" href="/products/mouse">View details</a>
</div>

<div class="product-card">
  <h2>Keyboard</h2>
  <p class="price">$45.00</p>
  <a class="product-link" href="/products/keyboard">View details</a>
</div>
"""

soup = BeautifulSoup(html, "html.parser")

products = []

for card in soup.select("___"):
    name = card.select_one("___").get_text(strip=True)
    price = card.select_one("___").get_text(strip=True)
    url = card.select_one("___").get("href")

    products.append({
        "name": name,
        "price": price,
        "url": url,
    })

print(products)

### Exercise B - Parse Inline HTML with Beautiful Soup

**Objective:** parse repeated product cards from an HTML string.

Loop over the product cards in `exercise_b_soup`. For each card, extract the name, price, and relative URL, then append a dictionary to `exercise_b_products` with exactly these keys: `name`, `price`, and `url`.

Hints:
- Loop over the parent cards first.
- Use `.select_one(...)` inside each card.
- Use `.get_text(strip=True)` for text fields.
- Use `.get("href")` for the URL.


In [ ]:

# Exercise B TODO

from bs4 import BeautifulSoup

exercise_b_html = """
<div class="product-card">
  <h2>Wireless Mouse</h2>
  <p class="price">$25.99</p>
  <a class="product-link" href="/products/mouse">View details</a>
</div>

<div class="product-card">
  <h2>Keyboard</h2>
  <p class="price">$45.00</p>
  <a class="product-link" href="/products/keyboard">View details</a>
</div>
"""

exercise_b_soup = BeautifulSoup(exercise_b_html, "html.parser")
exercise_b_products = []

# TODO: loop over product cards and append dictionaries to exercise_b_products


In [ ]:

# Exercise B Checks

assert isinstance(exercise_b_products, list), "exercise_b_products must be a list."
assert len(exercise_b_products) == 2, "You should extract exactly two products."
assert exercise_b_products[0]["name"] == "Wireless Mouse"
assert exercise_b_products[0]["price"] == "$25.99"
assert exercise_b_products[0]["url"] == "/products/mouse"
assert exercise_b_products[1]["name"] == "Keyboard"
assert set(exercise_b_products[0]) == {"name", "price", "url"}

print("Exercise B passed")


## Lab Exercise 1 - Requests and Beautiful Soup

Now use the selectors you practiced on the full local static page. This is the static-page workflow: fetch HTML, parse product cards, validate the extracted records, and prepare a clean DataFrame.

**Objective:** use Requests and Beautiful Soup to extract product names, prices, and detail links from `static_url`.

Your task:
1. Request `static_url` and call `raise_for_status()`.
2. Parse the response HTML with `BeautifulSoup`.
3. Loop over each repeated product card.
4. Extract the product `name`, `price`, and detail-page `url`.
5. Convert each relative product link to an absolute URL with `urljoin`.
6. Store dictionaries in `exercise_1_products`, build `exercise_1_df`, and run the check cell.

Hints:
- Use the same parent-container pattern from the inline HTML practice.
- Text fields usually use `.get_text(strip=True)`.
- Link destinations usually live in an attribute, not in visible text.
- Build one dictionary per product before converting the list into a DataFrame.


In [ ]:

# Exercise 1 TODO

# response = requests.get(___, timeout=10)
# response.raise_for_status()
# soup = BeautifulSoup(___, "html.parser")

exercise_1_products = []

# TODO, Remember the inline HTML practice.

exercise_1_df = pd.DataFrame(exercise_1_products)
exercise_1_df


### Exercise 1 Checks


In [ ]:

# Run after completing Exercise 1.

assert len(exercise_1_df) == 3
assert {"name", "price", "url"}.issubset(exercise_1_df.columns)
assert "Wireless Mouse" in set(exercise_1_df["name"])
assert exercise_1_df["url"].str.startswith("http://127.0.0.1").all()


### Exercise C - Build a Reusable Parser Function

**Objective:** create a function that receives HTML text and returns a clean DataFrame.

Requirements:
- Function name: `parse_products_html`
- Input: `html_text`
- Output: a `pandas.DataFrame`
- Required columns: `product_id`, `name`, `price`, `url`
- Use the repeated parent selector `.product-card`
- Read product ID from the `data-product-id` attribute

Hint: the product id is stored as an attribute on each product card.


In [ ]:

# Exercise C TODO

import pandas as pd
from bs4 import BeautifulSoup


def parse_products_html(html_text):
    # TODO: parse html_text and return a DataFrame
    pass


In [ ]:

# Exercise C Checks

sample_html = (DEMO_DIR / "products_static.html").read_text(encoding="utf-8")
exercise_c_df = parse_products_html(sample_html)

assert isinstance(exercise_c_df, pd.DataFrame), "Return a pandas DataFrame."
assert list(exercise_c_df.columns) == ["product_id", "name", "price", "url"]
assert len(exercise_c_df) == 3
assert exercise_c_df.loc[0, "product_id"] == "001"
assert exercise_c_df.loc[0, "name"] == "Wireless Mouse"
assert exercise_c_df["price"].str.startswith("$").all()
assert exercise_c_df["url"].str.startswith("/products/").all()

print("Exercise C passed")
exercise_c_df


### Exercise D - Validate and Save Processed Output

**Objective:** validate the parsed data and save it to CSV.

Use your parser to read the local static product HTML, create a products DataFrame, save it as a CSV, and assign both required variables:

1. Use `parse_products_html` to parse `products_static.html`.
2. Store the DataFrame in `student_products_df`.
3. Save the result to `PROCESSED_DIR / "student_products.csv"`.
4. Store the CSV path in `student_products_path`.

Hints:
- Read the local HTML from `DEMO_DIR / "products_static.html"`.
- Use `.to_csv(..., index=False)` so pandas does not write the row index.
- Run the check cell after saving to confirm the file and DataFrame match.


In [ ]:

# Exercise D TODO

student_products_path = None
student_products_df = None

# TODO: parse local HTML, save CSV, assign student_products_path and student_products_df


In [ ]:

# Exercise D Checks

assert student_products_path is not None, "Assign the CSV path to student_products_path."
assert Path(student_products_path).exists(), "The CSV file should exist on disk."
assert isinstance(student_products_df, pd.DataFrame), "student_products_df must be a DataFrame."
assert len(student_products_df) == 3
assert {"product_id", "name", "price", "url"}.issubset(student_products_df.columns)

loaded_student_products = pd.read_csv(student_products_path)
assert len(loaded_student_products) == 3
assert loaded_student_products["name"].tolist() == student_products_df["name"].tolist()

print("Exercise D passed")
loaded_student_products


## Lab Exercise 2 - Selenium Dynamic Page to CSV

This exercise introduces browser automation for a dynamic page. The workflow is open the page, wait for elements, interact with the search form, extract results, and save them.

**Objective:** use Selenium to open a dynamic page, wait for content, search, extract values, and save them to CSV.

Complete the TODO cell below by using Selenium to interact with the local dynamic product search page.

Your task:
1. Create a headless Chrome driver and open `dynamic_url`.
2. Wait for the search input, type `keyboard`, and click the search button.
3. Wait for the filtered product cards to appear.
4. For each card, extract the product `name`, `price`, and detail-page `url`.
5. Store the dictionaries in `selenium_results`, build `selenium_results_df`, and review the result.

Use explicit waits for elements you need to type into, click, or read.

**Expected output:** after searching for `keyboard`, the CSV should contain one product: Keyboard.

**Troubleshooting tips**
- If Chrome cannot start, check browser installation and Selenium version.
- If no elements are found, check the selector and wait condition.
- If the script hangs, reduce wait time and print progress logs.
- Always call `driver.quit()` in a `finally` block.


In [ ]:

# Exercise 2 TODO

# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC

selenium_results = []

# options = Options()
# options.add_argument("--headless=new")
# options.add_argument("--window-size=1200,900")
#
# driver = webdriver.Chrome(options=options)
# wait = WebDriverWait(driver, 10)
#
# try:
#     driver.get(dynamic_url)
#
#     search_box = wait.until(
#         EC.visibility_of_element_located((By.CSS_SELECTOR, "___"))
#     )
#     search_box.send_keys("keyboard")
#
#     button = wait.until(
#         EC.element_to_be_clickable((By.CSS_SELECTOR, "___"))
#     )
#     button.click()
#
#     cards = wait.until(
#         EC.presence_of_all_elements_located((By.CSS_SELECTOR, "___"))
#     )
#
#     for card in cards:
#         selenium_results.append({
#             "name": card.find_element(By.CSS_SELECTOR, "___").text,
#             "price": card.find_element(By.CSS_SELECTOR, "___").text,
#             "url": card.find_element(By.CSS_SELECTOR, "___").get_attribute("href"),
#         })
# finally:
#     driver.quit()

selenium_results_df = pd.DataFrame(selenium_results)
selenium_results_df


### Exercise 2 Checks


In [ ]:
# Run after completing Exercise 2.

if not selenium_results_df.empty:
    assert len(selenium_results_df) == 1
    assert selenium_results_df.loc[0, "name"] == "Keyboard"
    assert selenium_results_df.loc[0, "price"] == "$45.00"
    assert "url" in selenium_results_df.columns
    print("Exercise 2 passed")
else:
    print("Exercise 2 not completed yet, or Selenium could not run in this environment.")


### Optional Exercise E - Selenium Workflow Extension

**Objective:** turn the Selenium search workflow into a saved output.

Complete this only if Selenium and Chrome are available on your machine. This is intentionally more open-ended than Exercise 2.

Expected result after searching for `keyboard`:
- One row in `selenium_search_df`
- A saved CSV path in `selenium_search_path`
- Product name: `Keyboard`
- Price: `$45.00`

Hints:
- Use IDs for form controls when stable IDs exist.
- Use the same product-card extraction pattern after results appear.
- Use `try/finally` so the browser closes even when an error happens.
- Save to `PROCESSED_DIR / "selenium_search_results.csv"`.


In [ ]:
# Optional Exercise E TODO

selenium_search_df = pd.DataFrame()
selenium_search_path = None

# TODO if your environment supports Selenium:
# 1. Open dynamic_url in a browser driver.
# 2. Wait for the search input to be visible.
# 3. Type "keyboard" and click the search button.
# 4. Wait for the product cards to appear.
# 5. Extract product rows into selenium_search_df.
# 6. Save selenium_search_df to selenium_search_path.


In [ ]:
# Optional Exercise E Checks
# Run this only after implementing the Selenium TODO cell.

if not selenium_search_df.empty:
    assert len(selenium_search_df) == 1
    assert selenium_search_df.loc[0, "name"] == "Keyboard"
    assert selenium_search_df.loc[0, "price"] == "$45.00"
    assert selenium_search_path is not None
    assert Path(selenium_search_path).exists()
    print("Exercise E passed")
else:
    print("Exercise E skipped because selenium_search_df is empty.")


---

## Mini Project Starter - Simple Web Ingestion Workflow

This is a starter example for the later mini-project section. You do not need to finish the full project right now.

For now, use it to see how a small ingestion project can be organized. The starter files give you a place to begin building an extract, parse, validate, save, and log workflow in the next section.

Later, you can expand it into this architecture:

```text
Website -> Requests/Selenium -> Parser -> Validation -> CSV -> Log summary
```

Starter goals for later:
1. Extract data from the local website.
2. Parse relevant fields.
3. Store results in CSV.
4. Add logging.
5. Add basic error handling.
6. Add a simple validation check.


## Suggested Project Folder Structure

A small project becomes easier to debug when raw data, processed data, logs, source code, and documentation have predictable locations.

The next cell creates this starter structure and writes TODO comments into the source files.

```text
web_ingestion_project/
|-- data/
|   |-- raw/
|   `-- processed/
|-- logs/
|-- src/
|   |-- extract.py
|   |-- transform.py
|   |-- load.py
|   `-- main.py
|-- requirements.txt
`-- README.md
```


In [ ]:
project_root = BASE_DIR / "web_ingestion_project"
for path in [
    project_root / "data" / "raw",
    project_root / "data" / "processed",
    project_root / "logs",
    project_root / "src",
]:
    path.mkdir(parents=True, exist_ok=True)

starter_files = {
    "extract.py": "# TODO: write functions that fetch raw HTML or browser-rendered HTML.\n",
    "transform.py": "# TODO: write parser functions that turn raw HTML into rows.\n",
    "load.py": "# TODO: write functions that save DataFrames or records to disk.\n",
    "main.py": "# TODO: connect extract, transform, validate, load, and logging steps.\n",
}

for filename, content in starter_files.items():
    target = project_root / "src" / filename
    if not target.exists():
        target.write_text(content, encoding="utf-8")

(project_root / "requirements.txt").write_text("requests\nbeautifulsoup4\npandas\n", encoding="utf-8")
(project_root / "README.md").write_text(
    "# Web Ingestion Project\n\nStarter project for the later mini-project section.\n",
    encoding="utf-8",
)

print(project_root)
